
### Disease Onset - PLP Study

##### Research Question - For a given patient who is newly diagnosed with atrial fibrillation, what is the probability that they will go onto to have ischemic stroke in next 3 years?

##### Step 1 - Import the necessary libraries


In [ ]:
%%jupyter
# import libraries
library(rD2E)
library(Strategus)
library(dplyr)

##### Step 2 - Cohort definition loading

In [ ]:
%%jupyter
tarCohortId <- XX
outCohortId <- XX
cohorts_set <- c(tarCohortId, outCohortId)
cohortDefinitionSet <- rD2E::get_cohort_definition_set(cohorts_set)


##### Step 3 - Define the network study components

In [ ]:
%%jupyter
# Target: newly diagnosed AFib patients
targetCohortId <- tarCohortId  # Atrial Fibrillation
outcomeCohortId <- outCohortId  # Ischemic Stroke

# Time-at-risk: 3 years after diagnosis
timeAtRisks <- tibble(
  label = c("3-year stroke risk"),
  riskWindowStart = c(0),
  startAnchor = c("cohort start"),
  riskWindowEnd = c(1095),  # 3 years
  endAnchor = c("cohort start")
)

# Define the outcome
outcomeList <- list(
  CohortMethod::createOutcome(
    outcomeId = outcomeCohortId,
    outcomeOfInterest = TRUE
  )
)

# Define target-outcome structure (single-arm)
targetComparatorOutcomesList <- list(
  CohortMethod::createTargetComparatorOutcomes(
    targetId = targetCohortId,
    comparatorId = NA,
    outcomes = outcomeList
  )
)

Create settings for modules involved in the study

In [ ]:
%%Jupyter
# Setup cohort method module
cmModuleSettingsCreator <- CohortMethodModule$new()

cmAnalysisList <- list(
  CohortMethod::createCmAnalysis(
    analysisId = 1,
    description = "3-year ischemic stroke risk after AFib diagnosis",
    getDbCohortMethodDataArgs = CohortMethod::createGetDbCohortMethodDataArgs(
      studyStartDate = studyStartDate,
      studyEndDate = studyEndDate
    ),
    createStudyPopArgs = CohortMethod::createCreateStudyPopulationArgs(
      firstExposureOnly = TRUE,
      removeDuplicateSubjects = "keep first",
      removeSubjectsWithPriorOutcome = TRUE,
      priorOutcomeLookback = 0,
      requireTimeAtRisk = FALSE,
      riskWindowStart = timeAtRisks$riskWindowStart,
      startAnchor = timeAtRisks$startAnchor,
      riskWindowEnd = timeAtRisks$riskWindowEnd,
      endAnchor = timeAtRisks$endAnchor
    )
  )
)

cohortMethodModuleSpecifications <- cmModuleSettingsCreator$createModuleSpecifications(
  cmAnalysisList = cmAnalysisList,
  targetComparatorOutcomesList = targetComparatorOutcomesList
)


In [ ]:
%%jupyter
# Cohort Generator
cgModuleSettingsCreator <- CohortGeneratorModule$new()
cohortDefinitionShared <- cgModuleSettingsCreator$createCohortSharedResourceSpecifications(cohortDefinitionSet)
cohortGeneratorModuleSpecifications <- cgModuleSettingsCreator$createModuleSpecifications()

# Final Analysis Spec
analysisSpecifications <- createEmptyAnalysisSpecificiations() |>
  addSharedResources(cohortDefinitionShared) |>
  addModuleSpecifications(cohortGeneratorModuleSpecifications) |>
  addModuleSpecifications(cohortMethodModuleSpecifications)


Create the analysis specification object - including all the modules and configurations created above

In [ ]:
%%jupyter
# Create the analysis specifications ------------------------------------------
analysisSpecifications <- Strategus::createEmptyAnalysisSpecificiations() |>
  Strategus::addSharedResources(cohortDefinitionShared) |> 
  Strategus::addModuleSpecifications(cohortGeneratorModuleSpecifications) |>
  Strategus::addModuleSpecifications(plpModuleSpecifications)

##### Step 4 - Execute the Strategus study

In [ ]:
%%jupyter
# temporary workaround for running strategus on d2e server
executionSettings <- Strategus::createCdmExecutionSettings(
    workDatabaseSchema = "cdm_5pct_9a0f90a32250497d9483c981ef1e1e70",
    cdmDatabaseSchema = "cdm_5pct_9a0f90a32250497d9483c981ef1e1e70",
    cohortTableNames = CohortGenerator::getCohortTableNames(cohortTable = "cohort"),
    workFolder = "/tmp/<notebook_name>/work",
    resultsFolder = "/tmp/<notebook_name>/results",
    logFileName = "/tmp/<notebook_name>/strategus-log.txt",
    minCellCount = 5,
    maxCores = 8,
  )

In [ ]:
%%jupyter
study_name <- "treatment_safety_study"  # Unique study name
options <- create_options(upload_results=TRUE, study_id = study_name) # set a study_id with a unique id
options$studyId <- study_name
rD2E::run_strategus_flow(analysisSpecification = analysisSpecifications, options = options)